# Риск задержки > 150 с: 13 ТС, CatBoost, LightGBM и признаки движения
Сравнение проводится на реальных точках 13 отложенных исходных ТС. Синтетические копии отложенного ТС исключаются из обучения. Архив копируется с Drive в `/content` перед чтением.

In [ ]:
# Укажите свои пути на Google Drive.
DRIVE_DATASET_ZIP = '/content/drive/MyDrive/dataset.zip'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/mos-trans/risk-comparison'
BRANCH = 'ya-dolbayob'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from shutil import copy2
import hashlib, os, subprocess, sys
source_zip = Path(DRIVE_DATASET_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(source_zip)
local_zip = Path('/content/dataset.zip')
copy2(source_zip, local_zip)
with local_zip.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()[:16]
print('Archive hash:', digest)
repo = Path('/content/Mos-TRANS')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/epitaph76/Mos-TRANS.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements-risk.txt')], check=True)
os.chdir(repo)
revision = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Commit:', revision)


In [ ]:
from mos_trans.modeling.compare_risk import run
result_dir = Path(DRIVE_RESULTS_DIR) / f'{digest}-{revision}'
result_dir.mkdir(parents=True, exist_ok=True)
report = run(local_zip, Path('/content/processed'), result_dir / 'feature-cache', result_dir)
print('Selected:', report['selected_by_oof_ap'])
print('Results:', result_dir)


In [ ]:
import pandas as pd
display(pd.DataFrame(report['comparison']).T[['ap', 'f1', 'precision', 'recall', 'oof_threshold']].sort_values('ap', ascending=False))
display(pd.DataFrame(report['oof_by_family']).T[['n', 'positives', 'ap', 'f1']])
display(pd.DataFrame([report['test']], index=['published test']))
print('Model:', result_dir / 'selected_model.joblib')
print('Holdout and test predictions:', result_dir / 'oof_predictions.csv', result_dir / 'test_predictions.csv')
